# DSA Week 2 -- Benchmark Literacy

**Course:** Data Structures & Algorithms (Year 2, Semester 4)
**Session:** 3 hours
**Prerequisites:** Week 1 (Big-O)
**Focus:** timeit, scaling plots, measuring performance properly

## Learning Objectives

By the end of this session you will be able to:

1. Use `timeit` for accurate micro-benchmarks
2. Avoid common benchmarking pitfalls (warmup, garbage collection, outliers)
3. Run a scaling test: measure at multiple input sizes
4. Create publication-quality benchmark plots
5. Interpret results to confirm Big-O predictions
6. Build a reusable benchmark harness for your project

## Why This Week Matters

Last week you learned to *predict* performance with Big-O. This week you learn
to *measure* it. In engineering, if you cannot measure it, you cannot improve it.
Your final project requires a benchmark proving >= 1.5x speedup. Today you build
the tools to do that.

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Why `time.time()` Is Not Enough

Last week we used `time.time()` for quick demos. But for real benchmarks,
it has problems:

```
Problems with time.time():
  1. Measures wall clock -- includes OS interrupts, other programs
  2. Single measurement -- could be an outlier
  3. Low resolution on some systems -- cannot measure microseconds
  4. No warmup -- first run is often slower (CPU cache, JIT)
```

Python's `timeit` module fixes all of these:
- Runs the code many times and averages
- Disables garbage collection during measurement
- Uses the highest-resolution timer available

In [ ]:
import timeit

# === Basic timeit usage ===

# Method 1: timeit.timeit with a lambda
time_taken = timeit.timeit(lambda: sum(range(1000)), number=10_000)
print("sum(range(1000)) x 10,000 runs: " + "{:.4f}".format(time_taken) + "s")
print("Average per call: " + "{:.2f}".format(time_taken / 10_000 * 1e6) + " us")
print()

# Method 2: comparing two approaches
def slow_sum(data):
    total = 0
    for x in data:
        total += x
    return total

def fast_sum(data):
    return sum(data)

data = list(range(10_000))
n_runs = 1000

t_slow = timeit.timeit(lambda: slow_sum(data), number=n_runs)
t_fast = timeit.timeit(lambda: fast_sum(data), number=n_runs)

print("=== Loop sum vs built-in sum (n=10,000) ===")
print("  Loop sum:     " + "{:.4f}".format(t_slow) + "s (" + str(n_runs) + " runs)")
print("  Built-in sum: " + "{:.4f}".format(t_fast) + "s (" + str(n_runs) + " runs)")
print("  Speedup:      " + "{:.1f}".format(t_slow / t_fast) + "x")
print()
print("  Per-call average:")
print("    Loop:     " + "{:.2f}".format(t_slow / n_runs * 1e6) + " us")
print("    Built-in: " + "{:.2f}".format(t_fast / n_runs * 1e6) + " us")

**Expected Output:**
```
sum(range(1000)) x 10,000 runs: 0.4523s
Average per call: 45.23 us

=== Loop sum vs built-in sum (n=10,000) ===
  Loop sum:     3.2145s (1000 runs)
  Built-in sum: 0.2134s (1000 runs)
  Speedup:      15.1x

  Per-call average:
    Loop:     3214.50 us
    Built-in:  213.40 us
```

---
## Part 2: Scaling Tests -- Confirming Big-O Experimentally

A scaling test measures execution time at **multiple input sizes** and checks
whether the growth matches your Big-O prediction.

```
If you predict O(n):
  Double the input -> time should roughly double
  10x the input   -> time should roughly 10x

If you predict O(n^2):
  Double the input -> time should roughly 4x (2^2)
  10x the input   -> time should roughly 100x (10^2)

If you predict O(log n):
  Double the input -> time should increase by a constant
  10x the input   -> time should increase by a constant
```

In [ ]:
import timeit

def benchmark_scaling(func, sizes, n_runs=100, label=""):
    """Measure function time across different input sizes."""
    results = []
    for n in sizes:
        data = list(range(n))
        t = timeit.timeit(lambda d=data: func(d), number=n_runs)
        avg_ms = t / n_runs * 1000
        results.append({"n": n, "time_ms": avg_ms})
    return results

def print_scaling_table(results, label):
    """Print a formatted scaling table with growth ratios."""
    print("=== " + label + " ===")
    print("  " + "n".rjust(10) + " | " + "time (ms)".rjust(10) + " | " + "ratio".rjust(8))
    print("  " + "-" * 10 + "-|-" + "-" * 10 + "-|-" + "-" * 8)
    for i, r in enumerate(results):
        ratio = ""
        if i > 0 and results[i - 1]["time_ms"] > 0:
            ratio = "{:.1f}x".format(r["time_ms"] / results[i - 1]["time_ms"])
        print("  " + "{:>10,}".format(r["n"]) + " | " + "{:>10.3f}".format(r["time_ms"]) + " | " + ratio.rjust(8))
    print()

# Test 1: O(n) -- sum
sizes = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000]
results_sum = benchmark_scaling(sum, sizes, n_runs=200, label="sum")
print_scaling_table(results_sum, "sum() -- expected O(n)")
print("  If O(n): doubling n should roughly double the time.")
print("  Check the 1000->2000 ratio and 10000->20000 ratio.")
print()

# Test 2: O(n^2) -- nested membership
def quadratic_example(data):
    count = 0
    for x in data:
        if x in data:
            count += 1
    return count

sizes_q = [500, 1_000, 2_000, 4_000]
results_q = benchmark_scaling(quadratic_example, sizes_q, n_runs=5, label="quadratic")
print_scaling_table(results_q, "quadratic -- expected O(n^2)")
print("  If O(n^2): doubling n should roughly 4x the time.")

**Expected Output** (times vary):
```
=== sum() -- expected O(n) ===
           n |  time (ms) |    ratio
  ---------- | ---------- | --------
       1,000 |      0.010 |
       2,000 |      0.020 |    2.0x
       5,000 |      0.050 |    2.5x
      10,000 |      0.100 |    2.0x
      20,000 |      0.200 |    2.0x
      50,000 |      0.500 |    2.5x

  If O(n): doubling n should roughly double the time.

=== quadratic -- expected O(n^2) ===
           n |  time (ms) |    ratio
  ---------- | ---------- | --------
         500 |      2.500 |
       1,000 |     10.000 |    4.0x
       2,000 |     40.000 |    4.0x
       4,000 |    160.000 |    4.0x

  If O(n^2): doubling n should roughly 4x the time.
```

**Reading the ratio column:**
- Ratios near 2.0x when doubling input = O(n)
- Ratios near 4.0x when doubling input = O(n^2)
- Ratios near 1.0x when doubling input = O(1) or O(log n)

---
## Part 3: Creating Benchmark Plots

A picture is worth a thousand numbers. Let us build a reusable plotting function.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

def plot_scaling(results_list, labels, title, savepath, expected_curves=None):
    """Create a benchmark scaling plot.

    results_list: list of result dicts [{n, time_ms}, ...]
    labels: list of labels for each series
    title: plot title
    savepath: where to save
    expected_curves: optional list of (label, func) for theoretical curves
    """
    colors = ["#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0"]
    fig, ax = plt.subplots(figsize=(10, 6))

    for i, (results, label) in enumerate(zip(results_list, labels)):
        ns = [r["n"] for r in results]
        times = [r["time_ms"] for r in results]
        color = colors[i % len(colors)]
        ax.plot(ns, times, "o-", label=label, linewidth=2, markersize=6, color=color)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Input Size (n)", fontsize=12)
    ax.set_ylabel("Time (ms)", fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    os.makedirs(os.path.dirname(savepath), exist_ok=True)
    fig.savefig(savepath, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)

# Demo: plot linear vs quadratic scaling
plot_scaling(
    [results_sum, results_q],
    ["sum() O(n)", "quadratic O(n^2)"],
    "Scaling Test: O(n) vs O(n^2)",
    "reports/benchmark/scaling_test.png"
)

**Expected Output:**
```
Saved: reports/benchmark/scaling_test.png
```

---
## Part 4: Baseline vs Optimized Comparison

This is the pattern you will use for your project: measure baseline, measure
optimized, compute speedup, plot both.

In [ ]:
import timeit
import json

def benchmark_comparison(baseline_func, optimized_func, data_generator,
                         sizes, n_runs=100, label="Benchmark"):
    """Run a complete baseline vs optimized benchmark."""
    results = {
        "sizes": [],
        "baseline_ms": [],
        "optimized_ms": [],
        "speedup": []
    }

    print("=== " + label + " ===")
    print("  " + "n".rjust(10) + " | " + "baseline".rjust(10) + " | " + "optimized".rjust(10) + " | " + "speedup".rjust(8))
    print("  " + "-" * 10 + "-|-" + "-" * 10 + "-|-" + "-" * 10 + "-|-" + "-" * 8)

    for n in sizes:
        data = data_generator(n)
        t_base = timeit.timeit(lambda d=data: baseline_func(d), number=n_runs)
        t_opt = timeit.timeit(lambda d=data: optimized_func(d), number=n_runs)

        base_ms = t_base / n_runs * 1000
        opt_ms = t_opt / n_runs * 1000
        speedup = base_ms / opt_ms if opt_ms > 0 else float("inf")

        results["sizes"].append(n)
        results["baseline_ms"].append(round(base_ms, 4))
        results["optimized_ms"].append(round(opt_ms, 4))
        results["speedup"].append(round(speedup, 1))

        print("  " + "{:>10,}".format(n) + " | " + "{:>8.3f}ms".format(base_ms) + " | " + "{:>8.3f}ms".format(opt_ms) + " | " + "{:>6.1f}x".format(speedup))

    print()
    return results

def plot_comparison(results, title, savepath):
    """Create baseline vs optimized comparison plot."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    sizes = results["sizes"]

    # Left: time comparison
    ax1.plot(sizes, results["baseline_ms"], "o-", label="Baseline",
             linewidth=2, color="#F44336")
    ax1.plot(sizes, results["optimized_ms"], "s-", label="Optimized",
             linewidth=2, color="#4CAF50")
    ax1.set_title("Execution Time", fontsize=14)
    ax1.set_xlabel("Input Size (n)")
    ax1.set_ylabel("Time (ms)")
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)

    # Right: speedup bars
    ax2.bar(range(len(sizes)), results["speedup"], color="#2196F3")
    ax2.set_xticks(range(len(sizes)))
    ax2.set_xticklabels(["{:,}".format(n) for n in sizes], rotation=45)
    ax2.set_title("Speedup Factor", fontsize=14)
    ax2.set_xlabel("Input Size (n)")
    ax2.set_ylabel("Speedup (x)")
    ax2.axhline(y=1.5, color="red", linestyle="--", label="1.5x target")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(os.path.dirname(savepath), exist_ok=True)
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)

# Demo: list search vs set search
import random
random.seed(42)

def gen_data(n):
    return [random.randint(0, n * 10) for _ in range(n)]

def baseline_search(data):
    return 999 in data

def optimized_search(data):
    s = set(data)
    return 999 in s

sizes = [1_000, 5_000, 10_000, 50_000, 100_000]
results = benchmark_comparison(baseline_search, optimized_search, gen_data,
                               sizes, n_runs=50, label="List search vs Set")

plot_comparison(results, "Baseline (list) vs Optimized (set)",
                "reports/benchmark/comparison_demo.png")

# Save as JSON
os.makedirs("reports/benchmark", exist_ok=True)
with open("reports/benchmark/comparison_demo.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved: reports/benchmark/comparison_demo.json")

**Expected Output:**
```
=== List search vs Set ===
           n |   baseline |  optimized |  speedup
  ---------- | ---------- | ---------- | --------
       1,000 |    0.010ms |    0.025ms |    0.4x
       5,000 |    0.050ms |    0.120ms |    0.4x
      10,000 |    0.100ms |    0.240ms |    0.4x
      50,000 |    0.500ms |    1.200ms |    0.4x
     100,000 |    1.000ms |    2.400ms |    0.4x

Saved: reports/benchmark/comparison_demo.png
Saved: reports/benchmark/comparison_demo.json
```

**Wait -- the optimized version is SLOWER?** Yes! Building the set costs O(n),
so for a SINGLE search, list is fine. The optimization only helps when you
search MANY times. That is an important lesson: **always benchmark!**

---
## Part 5: Benchmarking Pitfalls

### Pitfall 1: Not enough runs
```python
# BAD: single measurement -- could be an outlier
t = timeit.timeit(func, number=1)

# GOOD: many runs
t = timeit.timeit(func, number=1000)
```

### Pitfall 2: Measuring setup time
```python
# BAD: includes data creation in the timing
t = timeit.timeit(lambda: sum(list(range(10000))), number=1000)

# GOOD: create data once, measure only the operation
data = list(range(10000))
t = timeit.timeit(lambda: sum(data), number=1000)
```

### Pitfall 3: Not checking correctness first
```
Always verify that baseline and optimized produce the SAME result
before comparing speed. A fast wrong answer is useless.
```

### Pitfall 4: Comparing at only one size
```
O(n) and O(n^2) look similar at small n. You need at least 3-5
different sizes to see the growth pattern.
```

### Try It Yourself

Build a benchmark that compares `list.sort()` vs `sorted()` at 5 different sizes.
Plot the results. What Big-O do you observe for both?

In [ ]:
# TODO: Benchmark list.sort() vs sorted()
import timeit
import random

def bench_sort_inplace(data):
    d = data.copy()
    d.sort()
    return d

def bench_sort_new(data):
    return sorted(data)

# Generate random data at different sizes
sizes = [1_000, 5_000, 10_000, 50_000, 100_000]

# YOUR CODE: benchmark both, print results, plot
# Hint: use benchmark_comparison from above

---
## Homework Preview

This week's homework asks you to build a complete benchmark suite for your
project track and produce benchmark_results.json and benchmark_plot.png.

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)